[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Computer Vision - Multi Modal Models

This notebooks shows how to use Large Vision Model for meta data extraction from a football game frame.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 13/09/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0130DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing
import imageio.v3 as iio

# Miscellaneous
import os
import random

# Typing
from typing import Callable, Dict, Literal, Optional, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython
from IPython.display import Markdown, display

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from TimeExtractor import LLMSession

In [ ]:
# General Auxiliary Functions

def ParseEnvFile( fileName: str = '.env', *, filePath: str = '.', keyValSep: str = '=' ) -> Dict[str, str]:
    # Read the file and parse it into a dictionary
    dEnv = {}
    with open(os.path.join(filePath, fileName), 'r') as hF:
        for line in hF:
            if line.startswith('#') or not line.strip():
                continue
            key, value = line.strip().split(keyValSep, 1)
            dEnv[key]  = value.strip()
    return dEnv

## Multi Modal Large Models

Many current LLMs are _Multi Modal Models_.  
A _Modality_ is a type of data: _Text_, _Image_, _Audio_ and _Video_.

A multi modal model can process more than one modality.  
For example, it can receive an image together with a text question.  
It can then answer the question using information from the image.

### Combining Modalities

Each modality must be converted into a numerical representation.  
An image encoder extracts visual features.  
Text is represented using token embeddings.  
The model combines these representations to reason about the inputs.

One common approach maps different modalities into a _Shared Embedding Space_.  
Related content can then have similar representations.  
For example, an _image of a football_ and the word _"football"_ can be aligned.  

* <font color='brown'>(**#**)</font> [OpenAI Developers page shows a comparisons of models](https://developers.openai.com/api/docs/models/compare) with their supported modalities.

### LLM API

Most LLM APIs use 2 main message types:

 - System Prompt: Defines the model's role, rules, and response style.
 - User Prompt: Provides the task and its data. It may include text or an image.

Model usage may be billed in 4 categories:

 - Input: New input processed by the model. Includes instructions and task data.
 - Output: Content generated by the model.
 - Cached Input: Previously cached prompt content reused by the model. Usually costs less.
 - Cached Write: Storing prompt content in a cache for later reuse.

A _Session_ groups related requests and their conversation history.  
Each request uses a chosen **model** and, when supported, an **effort level**.  
The model determines the maximum **context window** size.  
The same _system message_ is usually kept throughout the session.

* <font color='brown'>(**#**)</font> The context window is the maximum token budget for one request.  
  In a self attention layer, each token has a _Query_, a _Key_, and a _Value_.  
  During generation, the current token's query is compared with keys from earlier tokens and itself.  
  These matches determine the weights used to combine their values.  
  The window budget includes input and generated tokens.
* <font color='brown'>(**#**)</font> The effort level is a setting for how much reasoning a supported _Thinking Model_ performs before answering.  
  The model can generate intermediate reasoning steps, called _Chain of Thought_.  
  These steps help break a task into smaller parts. They may remain hidden from the user.  
  Higher effort usually allows more reasoning tokens. It can increase response time and cost.  
  More reasoning does not guarantee a correct answer.
* <font color='brown'>(**#**)</font> Text usage is measured in _Tokens_, not words. A token may be part of a word. Images also contribute to input usage.
* <font color='brown'>(**#**)</font> Prices depend on the model and provider. Not all APIs charge separately for cache writes.

## Structured Response

A _Structured Response_ follows a predefined _Schema_.  
The schema defines field names, data types, and allowed values.  
The model returns data, usually as _JSON_, instead of free text.

### Motivation

 - Reduce Hallucinations: Limit the response to the requested fields.  
   Allow `null` for missing or unreadable data instead of forcing a guess.
 - API Integration: Match the fields and types expected by other structured APIs.
 - Reliable Processing: Read values directly. Avoid extracting information from prose.
 - Validation: Check required fields, types, and allowed values before using the response.
 - Consistency: Use the same structure for every frame. Store and compare results easily.
 - Costs: Predictable cost per answer.

### Example

The task is to update a match tracking system from a football frame.  
Its API expects named fields for the time, score, added time, and match period.  
The score follows the _left to right_ order of the scoreboard.

#### Regular Response

The input prompt: "Read the match time, score, added time, and period from this frame."

The model might answer:

> The clock shows 90:23 in the second half. The score is 2-1, left to right. Five minutes of added time are indicated.

A person can read this easily. The API cannot use the sentence directly.  
Our code must extract the values and map them to the API's fields.  
The next response might use different wording. Our parsing logic must handle it.

#### Structured Response

We ask for the same information, using a predefined response structure:

```json
{
  "matchTime": "90:23",
  "score": {"left": 2, "right": 1},
  "addedTimeMinutes": 5,
  "period": "second_half"
}
```

* <font color='brown'>(**#**)</font> The structure can define what type of values are valid (Numbers, text, choice, etc...).
* <font color='brown'>(**#**)</font> Asking for JSON in a prompt is not the same as enforcing a schema.  
Structured response must be defined by a schema with supported API.
* <font color='brown'>(**#**)</font> A valid structure does not guarantee correct facts. Yet it might reduce the hallucinations.

In [ ]:
# Parameters

# Data
dataDirName = 'TimeExtractFootballFrames'
lImgFileExt = ['.jpg', '.png']

# Environment
envfile = '.env'

# Model
apiKey    = 'OPENAI_API_KEY'
modelName = 'gpt-6-luna' #<! GPT 6 Luna

# Visualization
numImg = 4

## The Task: Extracting Metadata from a Football Game Frame

![](https://i.imgur.com/uILF2bF.png)
<!-- ![](https://i.postimg.cc/KjMjZXqg/Frame00029.png) -->

Commonly, the scoreboard in a _football game_ (Soccer) frame shows:
 - Clock: The displayed match time. Usually minutes and seconds, such as `67:42`.
 - Result: The current number of goals scored by each team, such as `2 - 1`.
 - Added Time / Stoppage Time: Time to compensate for stoppages at the end of a period. An indicator such as `+5` means at least 5 added minutes.
 - Extra Time: Knockout matches use it when the score is tied after normal time. Usually 2 halves of 15 minutes.

The task is to read these values from the frame. The required data:

 - The match period (First half, second half, or either half of extra time).
 - The time.
 - The result.
 - The _Stoppage Time_.

* <font color='red'>(**?**)</font> Think of valid values per filed.
* <font color='brown'>(**#**)</font> A _Structured Response_ is well suited for the task above. 
* <font color='brown'>(**#**)</font> The response should handle the cases the model can not extract the required metadata.

## Generate / Load Data


In [ ]:
# Generate / Load Data

dataDirPath = os.path.join(DATA_FOLDER_PATH, dataDirName)

lImgFile = os.listdir(dataDirPath)
lImgFile = [f for f in lImgFile if os.path.splitext(f)[1].lower() in lImgFileExt]
numSamples = len(lImgFile)

print(f'Number of images: {numSamples}')

In [ ]:
# Environment

dEnv = ParseEnvFile(envfile)

### Plot the Data

In [ ]:
# Plot the Data

hF, vHa = plt.subplots(nrows = 3, ncols = 3, figsize = (12, 8))
vHa = vHa.flat

for ii, hA in enumerate(vHa):
    randIdx = random.randint(0, numSamples - 1)
    imgFileName = lImgFile[randIdx]
    imgFilePath = os.path.join(dataDirPath, imgFileName)

    tI = iio.imread(imgFilePath)
    
    hA.imshow(tI)
    hA.set_title(f'Label: {imgFileName}')
    hA.axis('off')

hF.tight_layout()

* <font color='red'>(**?**)</font> Is the whole image needed in order to accomplish the task? Think about optimizing the cost.

## Prose Session


In [ ]:
# Session

oSession = LLMSession(dEnv[apiKey], modelName)

In [ ]:
# Load Image

imgIdx = 123
imgFileName = lImgFile[imgIdx]
imgFilePath = os.path.join(dataDirPath, imgFileName)
tI = iio.imread(imgFilePath)

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tI)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Pre Process Image

numRows, numCols, numChannels = tI.shape

tX = tI[:(numRows // 2), :(numCols // 2), :]
tX = sp.ndimage.zoom(tX, (0.5, 0.5, 1.0))

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tX)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Prompt

promptMsg = "The input image is a frame from a video of a football game. The task is to extract the match period, the time. the score, the stoppage time and teh time within the stoppage time."

In [ ]:
# Answer

answerMsg = oSession.Prompt(promptMsg, tX)
display(Markdown(answerMsg))

* <font color='brown'>(**#**)</font> Modern day LLM's are so good they "predict" the intention.

## Structured Session

In [ ]:
# Response Schema





In [ ]:
# Iterate on the Loader

tX, vY = next(iter(dlTrain))
print(f'The batch image dimensions: {tX.shape}')
print(f'The batch label dimensions: {vY.shape}')

## Build Diffusion Model

The denoiser is a small convolutional _U-Net_: $28\times28 \rightarrow 14\times14 \rightarrow 7\times7$ and back.

 - Low resolution features capture the digit's overall structure.
 - Skip connections preserve spatial detail.
 - A sinusoidal time embedding tells each block the noise level.
 - The output is the predicted noise, with the same shape as the image.

* <font color='brown'>(**#**)</font> There is no classifier or sigmoid: Gaussian noise is not restricted to $[0,1]$.
* <font color='brown'>(**#**)</font> GroupNorm does not depend on batch statistics. Attention and a ViT are unnecessary for this small example.

```mermaid
flowchart TB
    Input["Noisy image: 1 x 28 x 28"] --> Enc1["Encoder 1: 32 x 28 x 28"]
    Enc1 -->|"Average pool"| Enc2["Encoder 2: 64 x 14 x 14"]
    Enc2 -->|"Average pool"| Mid["Middle: 128 x 7 x 7"]
    Mid -->|"Upsample"| Dec2["Decoder 2: 64 x 14 x 14"]
    Enc2 -->|"Skip: concatenate"| Dec2
    Dec2 -->|"Upsample"| Dec1["Decoder 1: 32 x 28 x 28"]
    Enc1 -->|"Skip: concatenate"| Dec1
    Dec1 -->|"1 x 1 convolution"| Output["Predicted noise: 1 x 28 x 28"]
    Time["Time step"] --> Embed["Sinusoidal embedding + MLP"]
    Embed -.-> Enc1
    Embed -.-> Enc2
    Embed -.-> Mid
    Embed -.-> Dec2
    Embed -.-> Dec1
```

Shapes are `channels x height x width` for `baseCh = 32`. Each encoder, middle, and decoder block receives the same time embedding through its own projection.

In [ ]:
# Time Conditioned U-Net

class TimeBlock(nn.Module):
    def __init__( self, inCh: int, outCh: int, timeDim: int ) -> None:
        super().__init__()
        self.oConv = nn.Sequential(nn.Conv2d(inCh, outCh, 3, padding = 1), nn.GroupNorm(8, outCh), nn.SiLU())
        self.oTime = nn.Linear(timeDim, outCh)
        self.oOut = nn.Sequential(nn.GroupNorm(8, outCh), nn.SiLU(), nn.Conv2d(outCh, outCh, 3, padding = 1))
        self.oSkip = nn.Conv2d(inCh, outCh, 1) if inCh != outCh else nn.Identity()

    def forward( self, tX: Tensor, mTime: Tensor ) -> Tensor:
        tZ = self.oConv(tX) + self.oTime(mTime)[:, :, None, None]
        return self.oOut(tZ) + self.oSkip(tX)

class DiffusionUNet(nn.Module):
    def __init__( self, baseCh: int = 32, timeDim: int = 64 ) -> None:
        super().__init__()
        vFreq = torch.exp(-np.log(10000.0) * torch.arange(timeDim // 2) / (timeDim // 2 - 1))
        self.register_buffer('vFreq', vFreq)
        self.oTime = nn.Sequential(nn.Linear(timeDim, timeDim), nn.SiLU(), nn.Linear(timeDim, timeDim))
        self.oEnc1 = TimeBlock(1, baseCh, timeDim)
        self.oEnc2 = TimeBlock(baseCh, 2 * baseCh, timeDim)
        self.oMid = TimeBlock(2 * baseCh, 4 * baseCh, timeDim)
        self.oDec2 = TimeBlock(6 * baseCh, 2 * baseCh, timeDim)
        self.oDec1 = TimeBlock(3 * baseCh, baseCh, timeDim)
        self.oOut = nn.Conv2d(baseCh, 1, 1)

    def forward( self, tX: Tensor, vTime: Tensor ) -> Tensor:
        mAngles = vTime.float()[:, None] * self.vFreq[None, :]
        mTime = self.oTime(torch.cat((mAngles.sin(), mAngles.cos()), dim = 1))
        tEnc1 = self.oEnc1(tX, mTime)
        tEnc2 = self.oEnc2(F.avg_pool2d(tEnc1, 2), mTime)
        tMid = self.oMid(F.avg_pool2d(tEnc2, 2), mTime)
        tDec2 = F.interpolate(tMid, size = tEnc2.shape[-2:], mode = 'nearest')
        tDec2 = self.oDec2(torch.cat((tDec2, tEnc2), dim = 1), mTime)
        tDec1 = F.interpolate(tDec2, size = tEnc1.shape[-2:], mode = 'nearest')
        tDec1 = self.oDec1(torch.cat((tDec1, tEnc1), dim = 1), mTime)
        return self.oOut(tDec1)

In [ ]:
# The Model Object

oModel = DiffusionUNet(baseCh)

In [ ]:
# Model Summary

torchinfo.summary(oModel, input_data = (torch.randn(2, 1, 28, 28), torch.zeros(2, dtype = torch.long)),
                  col_names = ['kernel_size', 'output_size', 'num_params'], device = 'cpu',
                  row_settings = ['depth', 'var_names'])

In [ ]:
# Model Input / Output

with torch.no_grad():
    tTest = torch.randn(2, 1, 28, 28)
    vTestTime = torch.tensor([0, numDiffSteps - 1])
    tNoiseHat = oModel(tTest, vTestTime)
assert tNoiseHat.shape == tTest.shape
print(f'Noisy images: {tTest.shape}, Time steps: {vTestTime.shape}, Predicted noise: {tNoiseHat.shape}')

## Train the Model

Each update samples clean images, a time step per image, and Gaussian noise. The model predicts the sampled noise:

$$ \mathcal{L}(\boldsymbol{w}) = \mathbb{E}_{\boldsymbol{x}_0,t,\boldsymbol{\epsilon}}\left[\left\|\boldsymbol{\epsilon}_{\boldsymbol{w}}(\boldsymbol{x}_t,t)-\boldsymbol{\epsilon}\right\|_2^2\right]. $$

 - Training uses one network evaluation per image, not the entire diffusion chain.
 - Validation uses fixed time steps and noise draws for comparable losses.
 - CUDA uses mixed precision. The best validation checkpoint is saved.

* <font color='brown'>(**#**)</font> Five epochs are a starting budget, not a guaranteed runtime. Target 3-7 minutes on a CUDA GPU; use the measured epoch time to adjust `numEpochs`, then check sample quality. CPU training may take substantially longer.
* <font color='brown'>(**#**)</font> Low noise MSE is useful, but does not by itself guarantee good generated images.

```mermaid
flowchart TB
    Clean["Clean images"] --> AddNoise["Forward formula"]
    Noise["Sample Gaussian noise"] --> AddNoise
    Time["Sample a time per image"] --> AddNoise
    AddNoise -->|"Noisy images"| Model["Time-conditioned U-Net"]
    Time --> Model
    Model -->|"Predicted noise"| Loss["MSE"]
    Noise -->|"Known target"| Loss
    Loss --> Update["Backpropagation + AdamW"]
    Update -.->|"Update weights"| Model
```

One randomly selected noise level per image; no reverse chain is needed during training.

In [ ]:
# Check GPU Availability

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device

In [ ]:
# Noise Prediction Loss

hL = nn.MSELoss()

* <font color='red'>(**?**)</font> Why can the model predict noise better than chance even though the noise was sampled independently of the clean image?
* <font color='brown'>(**#**)</font> Independence from the clean image does not imply independence from the noisy observation. The MSE optimum is $\mathbb{E}[\boldsymbol{\epsilon}\mid\boldsymbol{x}_t,t]$.

In [ ]:
# Training / Validation Epoch

def RunDiffusionEpoch( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hL: nn.Module, runDevice: torch.device, *, oOpt = None, oScaler = None, maxBatches: Optional[int] = None ) -> float:
    isTrain = oOpt is not None
    oModel.train(isTrain)
    oGen = None if isTrain else torch.Generator(device = runDevice).manual_seed(seedNum + 1)
    totalLoss = torch.zeros((), device = runDevice)
    numSeen = 0

    for batchIdx, (tX, _) in enumerate(dlData):
        if maxBatches is not None and batchIdx >= maxBatches:
            break
        tX = tX.to(runDevice, non_blocking = True)
        vTime = torch.randint(oDiff.numSteps, (len(tX),), device = runDevice, generator = oGen)
        tNoise = torch.randn(tX.shape, device = runDevice, generator = oGen)
        tNoisy = oDiff.AddNoise(tX, vTime, tNoise)
        if isTrain:
            oOpt.zero_grad(set_to_none = True)
        with torch.set_grad_enabled(isTrain):
            with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
                tNoiseHat = oModel(tNoisy, vTime)
                loss = hL(tNoiseHat.float(), tNoise)
            if isTrain:
                if oScaler is not None:
                    oScaler.scale(loss).backward()
                    oScaler.unscale_(oOpt)
                    nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
                    oScaler.step(oOpt)
                    oScaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
                    oOpt.step()
        totalLoss += loss.detach() * len(tX)
        numSeen += len(tX)

    return (totalLoss / numSeen).item()

In [ ]:
# Optimizer and Run Device

oModel = oModel.to(runDevice)
oDiff = oDiff.to(runDevice)
oOpt = torch.optim.AdamW(oModel.parameters(), lr = learnRate, weight_decay = 1e-4)
oScaler = torch.amp.GradScaler('cuda', enabled = runDevice.type == 'cuda')
modelPath = os.path.join(MODELS_FOLDER_PATH, modelName)

In [ ]:
# Training the Model

lTrainLoss, lValLoss, lEpochTime = [], [], []
bestLoss = float('inf')
os.makedirs(MODELS_FOLDER_PATH, exist_ok = True)

for epochIdx in range(numEpochs):
    startTime = perf_counter()
    trainLoss = RunDiffusionEpoch(oModel, oDiff, dlTrain, hL, runDevice, oOpt = oOpt, oScaler = oScaler)
    valLoss = RunDiffusionEpoch(oModel, oDiff, dlVal, hL, runDevice, maxBatches = numValBatches)
    elapsedTime = perf_counter() - startTime
    lTrainLoss.append(trainLoss)
    lValLoss.append(valLoss)
    lEpochTime.append(elapsedTime)
    if valLoss < bestLoss:
        bestLoss = valLoss
        torch.save({'Model': oModel.state_dict(), 'Diffusion': oDiff.state_dict(),
                    'BaseChannels': baseCh, 'NumSteps': numDiffSteps}, modelPath)
    print(f'Epoch {epochIdx + 1:02d}/{numEpochs}: Train MSE = {trainLoss:.4f}, Val MSE = {valLoss:.4f}, Time = {elapsedTime:.1f} s')

print(f'Total training and validation time: {sum(lEpochTime) / 60:.1f} minutes')

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(1, 2, figsize = (10, 4))
vEpoch = np.arange(1, len(lTrainLoss) + 1)
vHa[0].plot(vEpoch, lTrainLoss, label = 'Train')
vHa[0].plot(vEpoch, lValLoss, label = 'Validation')
vHa[0].set(title = 'Noise Prediction MSE', xlabel = 'Epoch', ylabel = 'MSE')
vHa[0].legend()
vHa[1].plot(vEpoch, lEpochTime)
vHa[1].set(title = 'Training and Validation Time', xlabel = 'Epoch', ylabel = 'Seconds')
hF.tight_layout();

In [ ]:
# Load the Best Model / Inference Mode

dModel = torch.load(modelPath, map_location = runDevice, weights_only = True)
if dModel['BaseChannels'] != baseCh or dModel['NumSteps'] != numDiffSteps:
    raise ValueError('Checkpoint configuration does not match the notebook parameters.')
oModel.load_state_dict(dModel['Model'])
oDiff.load_state_dict(dModel['Diffusion'])
oModel.eval();

### Generate by Repeated Denoising

```mermaid
flowchart TB
    Start["Sample Gaussian image; set t = T"] --> State["Current image and time"]
    State --> Model["U-Net: predict noise"]
    Model --> Step["Estimate clean image; compute reverse mean"]
    State --> Step
    Step --> Sample["Sample the next image"]
    Noise["Fresh Gaussian noise; zero at final step"] --> Sample
    Sample --> Done{"t = 1?"}
    Done -->|"No"| Next["Use next image; decrease t"]
    Next --> State
    Done -->|"Yes"| Output["Generated image"]
```

Weights stay fixed. The schedule scales the fresh noise by the reverse standard deviation; the final transition adds no noise. Here `t` uses the mathematical `T ... 1` convention; code indices are `T - 1 ... 0`.

In [ ]:
# Reverse Diffusion / DDPM Sampling

@torch.inference_mode()
def SampleDiffusion( oModel: nn.Module, oDiff: DiffusionSchedule, numSamples: int, runDevice: torch.device, sampleSeed: int = 512 ) -> Tuple[Tensor, list, list]:
    oModel.eval()
    oGen = torch.Generator(device = runDevice).manual_seed(sampleSeed)
    tX = torch.randn((numSamples, 1, 28, 28), device = runDevice, generator = oGen)
    lFrames = [tX[:4].cpu().clone()]
    lSteps = [oDiff.numSteps]
    vSaveSteps = np.linspace(oDiff.numSteps, 0, numPlotSteps, dtype = int)

    for stepIdx in reversed(range(oDiff.numSteps)):
        vTime = torch.full((numSamples,), stepIdx, device = runDevice, dtype = torch.long)
        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tNoiseHat = oModel(tX, vTime)
        tNoise = torch.randn(tX.shape, device = runDevice, generator = oGen) if stepIdx > 0 else torch.zeros_like(tX)
        tX = oDiff.Step(tX, tNoiseHat.float(), stepIdx, tNoise)
        if stepIdx in vSaveSteps:
            lFrames.append(tX[:4].cpu().clone())
            lSteps.append(stepIdx)

    return tX.cpu(), lFrames, lSteps

In [ ]:
# Generate New Images from Noise

tSamples, lFrames, lSteps = SampleDiffusion(oModel, oDiff, numImg, runDevice, sampleSeed = seedNum)
numCols = int(np.ceil(np.sqrt(numImg)))
numRows = int(np.ceil(numImg / numCols))
hF, mHa = plt.subplots(numRows, numCols, figsize = (2 * numCols, 2 * numRows), squeeze = False)
for sampleIdx, hA in enumerate(mHa.flat):
    if sampleIdx < numImg:
        hA.imshow(TensorImageNumpy(tSamples[sampleIdx]), cmap = 'gray', vmin = -1, vmax = 1)
    hA.axis('off')
hF.suptitle('Generated MNIST Images')
hF.tight_layout();

In [ ]:
# Follow the Reverse Process

numRows = len(lFrames[0])
hF, mHa = plt.subplots(numRows, len(lFrames), figsize = (2 * len(lFrames), 2 * numRows), squeeze = False)
for frameIdx, (tFrame, stepIdx) in enumerate(zip(lFrames, lSteps)):
    for sampleIdx in range(numRows):
        hA = mHa[sampleIdx, frameIdx]
        hA.imshow(TensorImageNumpy(tFrame[sampleIdx]), cmap = 'gray', vmin = -1, vmax = 1)
        hA.axis('off')
        if sampleIdx == 0:
            hA.set_title(f'Step {stepIdx}')
hF.tight_layout();

* <font color='blue'>(**!**)</font> Change `sampleSeed` and generate new images. No retraining is needed.
* <font color='red'>(**?**)</font> Why do we add noise during reverse sampling, but not at the final step?
* <font color='green'>(**@**)</font> Compare fixed-seed samples after different numbers of training epochs.
* <font color='green'>(**@**)</font> Remove the time embedding, retrain, and compare the results.
* <font color='brown'>(**#**)</font> Do not accelerate this DDPM sampler by simply skipping steps. A sampler such as DDIM uses a different update for larger jumps.